# 02 — EDA: Campaign Performance
Explore conversion rates by channel, month, contact frequency, and customer attributes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('../reports/figures', exist_ok=True)

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Shape: {df.shape}')
print(f'Conversion rate: {df["y"].mean()*100:.2f}%')
df.head(3)

## 1. Conversion Rate by Channel

In [ ]:
# Reconstruct contact column from dummies if needed
if 'contact_cellular' in df.columns:
    df['contact'] = np.where(df['contact_cellular'] == 1, 'cellular', 'telephone')

channel_conv = df.groupby('contact')['y'].agg(['mean','count','sum']).rename(
    columns={'mean':'conv_rate','count':'contacts','sum':'conversions'})
channel_conv['conv_rate_pct'] = channel_conv['conv_rate'] * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(channel_conv.index, channel_conv['conv_rate_pct'], color=['#4C72B0','#DD8452'], edgecolor='white')
for bar, val in zip(bars, channel_conv['conv_rate_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Conversion Rate by Contact Channel', fontsize=14)
ax.set_ylabel('Conversion Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('../reports/figures/channel_conversion.png', dpi=150)
plt.show()
print(channel_conv)

## 2. Conversion Rate by Month

In [ ]:
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_conv = df.groupby('month')['y'].mean().reindex(month_order) * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(month_conv.index, month_conv.values, marker='o', linewidth=2, color='#4C72B0')
ax.fill_between(month_conv.index, month_conv.values, alpha=0.15, color='#4C72B0')
ax.set_title('Monthly Conversion Rate Trend', fontsize=14)
ax.set_ylabel('Conversion Rate (%)')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig('../reports/figures/monthly_conversion.png', dpi=150)
plt.show()

## 3. Day of Week

In [ ]:
day_order = ['mon','tue','wed','thu','fri']
day_conv = df.groupby('day_of_week')['y'].mean().reindex(day_order) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(day_conv.index, day_conv.values, color='#55A868', edgecolor='white')
ax.set_title('Conversion Rate by Day of Week', fontsize=14)
ax.set_ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.savefig('../reports/figures/dow_conversion.png', dpi=150)
plt.show()
print(day_conv)

## 4. Contact Frequency Analysis

In [ ]:
freq_conv = df.groupby('campaign')['y'].mean() * 100
freq_conv = freq_conv[freq_conv.index <= 15]  # cap display at 15 contacts

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(freq_conv.index, freq_conv.values, color='#C44E52', edgecolor='white')
ax.set_title('Does More Contact Help? Conversion Rate by # of Contacts', fontsize=14)
ax.set_xlabel('Number of Contacts (this campaign)')
ax.set_ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.savefig('../reports/figures/contact_frequency.png', dpi=150)
plt.show()
print('Insight: Conversion rate typically peaks at 1-2 contacts, then drops sharply.')

## 5. Age Distribution — Converters vs Non-Converters

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df[df['y']==1]['age'].plot.hist(bins=30, alpha=0.6, label='Converted', ax=ax, color='#4C72B0')
df[df['y']==0]['age'].plot.hist(bins=30, alpha=0.6, label='Not Converted', ax=ax, color='#DD8452')
ax.set_title('Age Distribution: Converters vs Non-Converters', fontsize=14)
ax.set_xlabel('Age')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/age_distribution.png', dpi=150)
plt.show()

## 6. Campaign Funnel Chart

In [ ]:
total = len(df)
contacted_once = len(df[df['campaign'] >= 1])
contacted_twice = len(df[df['campaign'] >= 2])
converted = df['y'].sum()

stages = ['Total Records', 'Contacted ≥1x', 'Contacted ≥2x', 'Converted']
counts = [total, contacted_once, contacted_twice, converted]
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(stages[::-1], counts[::-1], color=colors[::-1], edgecolor='white')
for bar, val in zip(bars, counts[::-1]):
    ax.text(bar.get_width()*0.5, bar.get_y()+bar.get_height()/2,
            f'{val:,}  ({val/total*100:.1f}%)', va='center', color='white', fontweight='bold')
ax.set_title('Campaign Reach → Contact → Conversion Funnel', fontsize=14)
ax.set_xlabel('Count')
ax.set_xlim(0, total * 1.1)
plt.tight_layout()
plt.savefig('../reports/figures/campaign_funnel.png', dpi=150)
plt.show()